In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("white") 
sns.set_palette("pastel")
plt.rcParams.update({
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.figsize": (8, 4),
    "axes.edgecolor": "0.8",
})

df = pd.read_parquet("../data/processed/feature_dataset.parquet")
df.head()

df_model = pd.get_dummies(df, columns=["season", "movement_pattern"], drop_first=True)

# - Selección vars
target = "distance_per_day"
features = df_model.select_dtypes(include=[np.number]).drop(columns=[target, "UniqueAnimalID"]).columns.tolist()

df_model = df[[target] + features].dropna()

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

# Escalado y datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Diccionario de modelos y parámetros
modelos = {
    "Linear (scaled)": {
        "model": LinearRegression(),
        "params": {}
    },
    "RidgeCV (scaled)": {
        "model": Ridge(),
        "params": {
            "alpha": np.logspace(-3, 3, 20)
        }
    },
    "Random Forest": {
        "model": RandomForestRegressor(random_state=10),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20]
        }
    }
}

# Entrenamiento y evaluación
resultados = []

for nombre, m in modelos.items():
    print(f"Entrenando {nombre}...")

    if "scaled" in nombre:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test

    grid = GridSearchCV(m["model"], m["params"], cv=5, scoring="r2", n_jobs=-1)
    grid.fit(X_tr, y_train)
    best_model = grid.best_estimator_

    pred_train = best_model.predict(X_tr)
    pred_test = best_model.predict(X_te)

    resultados.append({
        "Modelo": nombre,
        "Mejor params": grid.best_params_,
        "CV mean R²": grid.best_score_,
        "MAE train": mean_absolute_error(y_train, pred_train),
        "MAE test": mean_absolute_error(y_test, pred_test),
        "RMSE train": np.sqrt(mean_squared_error(y_train, pred_train)),
        "RMSE test": np.sqrt(mean_squared_error(y_test, pred_test)),
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test)
    })

# Mostrar resultados ordenados
df_resultados = pd.DataFrame(resultados).sort_values("R² test", ascending=False)
print(df_resultados)

import pickle
#  Guardar mejor modelo
mejor_nombre = df_resultados.iloc[0]["Modelo"]
mejor_modelo = modelos[mejor_nombre]

with open("models/mejor_modelo_regresion_GS.pkl", "wb") as f:
    pickle.dump(mejor_modelo, f)

print(f"Modelo guardado: {mejor_nombre}")